# Visualising the QQQ Top-20 Max-Sharpe Back-test

Run this top to bottom, one cell at a time. Three sections, in the order the strategy
actually works:

1. **Stock selection** — what the screen kept out of the Nasdaq-100, and why
2. **Portfolio optimisation** — the efficient frontier, the tangency portfolio, and the weights it produced
3. **Profit and loss** — what that turned $10,000 into, against QQQ and BOXX

The strategy itself is **not defined here**. Everything comes from `qqq_maxsharpe.py`,
the same module `qqq_top20_max_sharpe_backtest.ipynb` runs — so these pictures are of
that back-test, not of a second copy that drifted from it. This notebook only draws.

Charts are Plotly, so they hover. The frontier-and-CML plot follows the shape of the
one in `M6_finalnotebook.ipynb`; the weight view uses a slider over bars rather than a
pie, because sixteen slices are not readable as a pie and the bars can be compared
against each other directly.

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import qqq_maxsharpe as qms

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
print(f"repo root: {REPO_ROOT}")

## 0 · House style

One place for colour and layout, so every chart below reads as the same system.

The three-hue categorical set is slots 1-3 of a palette validated for colour-vision
deficiency at all pairs — worst-pair ΔE 9.2 under deuteranopia, 24.0 under normal
vision. Aqua sits below 3:1 contrast on this surface, so every series that uses it also
carries a visible label or a table; identity is never colour alone. Magnitude uses one
blue ramp, light to dark. Polarity (profit against loss) uses blue against orange with a
neutral grey midpoint — never a rainbow, never a hue in the middle.

In [ ]:
# Categorical slots 1-3. Assigned by entity and never cycled: Strategy is always blue.
SERIES = {"Strategy": "#2a78d6", "QQQ": "#eb6834", "BOXX": "#1baf7a"}
POS, NEG = "#2a78d6", "#eb6834"          # diverging poles: gain / loss
SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
#: One hue, light to dark, for magnitude.
BLUE_RAMP = [[0.0, "#eef4fc"], [0.25, "#bcd6f4"], [0.5, "#7db0e8"], [0.75, "#4a8ddb"], [1.0, "#1c5ba8"]]
#: Two hues either side of a neutral grey, for polarity.
DIVERGING = [[0.0, "#c14a1c"], [0.25, "#f0a17f"], [0.5, "#eeedea"], [0.75, "#7db0e8"], [1.0, "#1c5ba8"]]


def figure(title: str, xtitle: str = "", ytitle: str = "", height: int = 460, **kw) -> go.Figure:
    """A figure with the recessive furniture already applied."""
    layout = dict(
        title=dict(text=title, x=0, xanchor="left", font=dict(size=16, color=INK)),
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
        font=dict(color=MUTED, size=12),
        xaxis=dict(title=xtitle, gridcolor=GRID, zeroline=False, linecolor=GRID,
                   ticks="outside", tickcolor=GRID),
        yaxis=dict(title=ytitle, gridcolor=GRID, zeroline=False, linecolor=GRID,
                   ticks="outside", tickcolor=GRID),
        legend=dict(bgcolor="rgba(0,0,0,0)", borderwidth=0),
        margin=dict(l=70, r=40, t=70, b=60),
        height=height, hovermode="closest",
    )
    layout.update(kw)          # caller wins, e.g. hovermode="x unified"
    fig = go.Figure()
    fig.update_layout(**layout)
    return fig


def banner(market) -> str:
    return "  ⚠ SYNTHETIC DATA" if market.synthetic else ""

## 1 · Run the pipeline

Three cells: settings, prices, back-test. The back-test cell is the slow one — it runs
the screen once per rebalance across the whole universe.

In [ ]:
CFG = qms.BacktestSettings()
UNIVERSE = qms.nasdaq100_symbols()
print(f"{len(UNIVERSE)} names in the universe | ${CFG.principal:,.0f} principal | "
      f"top {CFG.top_n} | window from {CFG.start}")

In [ ]:
MARKET = qms.load_market(UNIVERSE, CFG)

In [ ]:
EQUITY, TRADES, PLAN = qms.run_strategy(MARKET, UNIVERSE, CFG)
CURVES = qms.curves(MARKET, EQUITY, CFG)
REBALANCES = list(PLAN["rebalance"])
SIGNAL_DATES = list(PLAN["signal_date"])

print(f"{len(PLAN)} rebalances, {len(TRADES)} fills, "
      f"{CURVES.index[0].date()} -> {CURVES.index[-1].date()}{banner(MARKET)}")
PLAN[["rebalance", "signal_date", "n_passed", "n_held", "note"]].head()

---

# 2 · Stock selection

Everything in this section is the state of the screen on **one signal date**. Change
`LOOK_AT` to walk through the months — every chart below re-renders from it.

In [ ]:
# Which rebalance to inspect. -1 is the most recent; 0 is the first.
LOOK_AT = -1

SIGNAL = SIGNAL_DATES[LOOK_AT]
REBAL = REBALANCES[LOOK_AT]

# passed_only=False keeps the rejected names, which is what a funnel needs.
SCREEN = qms.screen_asof(MARKET, SIGNAL, UNIVERSE, CFG, passed_only=False)
PICKS = SCREEN[SCREEN["passed"]].head(CFG.top_n)["ticker"].tolist()

print(f"signal date {SIGNAL.date()}  ->  traded {REBAL.date()}")
print(f"{len(SCREEN)} names evaluated, {int(SCREEN['passed'].sum())} passed, "
      f"top {len(PICKS)} taken")
SCREEN.head(10)[["ticker", "last_close", "daily_annret", "ann_vol", "rs6m_vs_mkt",
                 "within_52w_high_pct", "passed"]]

### 2.1 · The funnel

How many names each gate removes. The gates are recovered from the fields
`ScreenResult` carries, in the order `evaluate_ticker` applies them; the last step adds
the Stage-2 trend template, which needs the 150-day SMA and its slope and so cannot be
split out of the result on its own.

A funnel is a magnitude chart, so it uses the single blue ramp — the colour carries "how
far down the funnel", not identity — with the count and the survival rate written on
each bar.

In [ ]:
# Cumulative, not independent: each gate is applied on top of the ones above it, so
# the bars can only shrink. Counting each test on the full set instead would let the
# funnel widen halfway down, which is not what a funnel means.
alive = pd.Series(True, index=SCREEN.index)
gates = [("In the universe", len(UNIVERSE)),
         ("Priced & liquid enough", int(alive.sum()))]
for label, test in (
    ("… above the 200-day SMA", SCREEN["last_close"] > SCREEN["sma200"]),
    ("… within 25% of 52w high", SCREEN["within_52w_high_pct"] <= 0.25),
    ("… beating the market 6m", SCREEN["rs6m_vs_mkt"] > 0),
    ("… + Stage-2 template = passed", SCREEN["passed"]),
):
    alive &= test
    gates.append((label, int(alive.sum())))
gates.append((f"Top {CFG.top_n} by annual return", len(PICKS)))
labels = [g[0] for g in gates][::-1]
counts = [g[1] for g in gates][::-1]
shades = ["#1c5ba8", "#2a78d6", "#4a8ddb", "#7db0e8", "#9dc4ee", "#bcd6f4", "#dbe9f9"]

fig = figure(f"What the screen removed — {SIGNAL.date()}{banner(MARKET)}",
             "Names surviving", height=430)
fig.add_bar(
    x=counts, y=labels, orientation="h", marker=dict(color=shades, line=dict(color=SURFACE, width=2)),
    text=[f"{c}  ({c / len(UNIVERSE):.0%})" for c in counts],
    textposition="outside", textfont=dict(color=INK, size=11),
    hovertemplate="%{y}<br>%{x} names<extra></extra>",
)
fig.update_xaxes(range=[0, len(UNIVERSE) * 1.28])
fig.update_yaxes(gridcolor=SURFACE)
fig.show()

### 2.2 · What got picked, in risk/return space

Every evaluated name plotted by trailing annualised volatility against trailing
annualised return — the two numbers the screen and the optimiser both care about. Grey
is rejected, blue is selected. Hover for the ticker and its relative strength.

The point of this chart is the shape of the selection: the screen is a momentum filter,
so the picks should sit high, and they will also sit right, because in this universe
return and volatility travel together. That is the bet the strategy is making, drawn
plainly.

In [ ]:
sel = SCREEN[SCREEN["ticker"].isin(PICKS)]
rej = SCREEN[~SCREEN["ticker"].isin(PICKS)]

fig = figure(f"Selected vs rejected — {SIGNAL.date()}{banner(MARKET)}",
             "Trailing annualised volatility", "Trailing annualised return", height=520)
for frame, name, color, size in ((rej, "Not selected", "#c8c7c1", 8), (sel, "Top 20", SERIES["Strategy"], 13)):
    fig.add_scatter(
        x=frame["ann_vol"], y=frame["daily_annret"], mode="markers", name=name,
        marker=dict(color=color, size=size, line=dict(color=SURFACE, width=1.5)),
        text=frame["ticker"], customdata=frame[["rs6m_vs_mkt", "within_52w_high_pct"]],
        hovertemplate=("<b>%{text}</b><br>return %{y:.1%}<br>vol %{x:.1%}"
                       "<br>RS vs mkt %{customdata[0]:+.1%}"
                       "<br>below 52w high %{customdata[1]:.1%}<extra></extra>"),
    )
# Direct labels, highest return first, skipping any that would land on one already
# placed. A label on all 20 would be a thicket, and nudging colliding labels apart just
# moves the ambiguity to which dot each one belongs to — hover names the rest.
x_span = max(SCREEN["ann_vol"].max() - SCREEN["ann_vol"].min(), 1e-9)
y_span = max(SCREEN["daily_annret"].max() - SCREEN["daily_annret"].min(), 1e-9)
placed = []
for _, r in sel.sort_values("daily_annret", ascending=False).iterrows():
    x, y = r["ann_vol"], r["daily_annret"]
    if any(abs(x - px) / x_span < 0.055 and abs(y - py) / y_span < 0.055 for px, py in placed):
        continue
    fig.add_annotation(x=x, y=y, text=r["ticker"], showarrow=False, yshift=16,
                       font=dict(size=10, color=INK))
    placed.append((x, y))
    if len(placed) == 10:
        break
fig.add_hline(y=0, line=dict(color=GRID, width=1))
fig.update_xaxes(tickformat=".0%")
fig.update_yaxes(tickformat=".0%")
fig.show()

### 2.3 · Turnover — does the screen keep changing its mind?

Which names were selected in which month. A dense row means a name the screen held on to;
a sparse column means a month where the selection churned. High turnover is not
automatically bad, but it is what the commission and slippage in section 4 are paying for.

In [ ]:
membership = pd.DataFrame(
    [{t: 1.0 for t in d if d[t] > 0} for d in PLAN["weights"]],
    index=[d.date() for d in REBALANCES],
).fillna(0.0)
# Most-held names first, so the chart reads top-down as "core" to "one-off".
membership = membership[membership.sum().sort_values(ascending=False).index]

held_months = membership.sum().astype(int)
churn = [int((membership.iloc[i] != membership.iloc[i - 1]).sum()) if i else np.nan
         for i in range(len(membership))]

fig = figure(f"Who was held, month by month{banner(MARKET)}", "", "",
             height=max(430, 21 * membership.shape[1]))
fig.add_heatmap(
    z=membership.T.values, x=[str(d) for d in membership.index], y=list(membership.columns),
    colorscale=[[0, "#f3f2ef"], [1, SERIES["Strategy"]]], showscale=False, xgap=2, ygap=2,
    hovertemplate="%{y} held on %{x}<extra></extra>",
)
# Category axes with dtick=1: every row keeps its ticker and every column its date.
# Left to itself Plotly thins the labels, and an unlabelled row cannot be read at all.
fig.update_yaxes(type="category", autorange="reversed", tickmode="linear", dtick=1,
                 gridcolor=SURFACE, tickfont=dict(size=10))
fig.update_xaxes(type="category", tickmode="linear", dtick=1, gridcolor=SURFACE, tickangle=-45)
fig.show()

print(f"{membership.shape[1]} distinct names over {len(membership)} rebalances")
print(f"median names changed per rebalance: {np.nanmedian(churn):.0f}")
held_months.head(12).to_frame("months held").T

---

# 3 · Portfolio optimisation result

The screen produced 20 candidates. The optimiser decides how much of each to own.
This section shows the problem it solved and the answer it gave, on the same
`LOOK_AT` date as section 2.

In [ ]:
RETS = qms.selected_returns(MARKET, SIGNAL, PICKS, CFG)
RF = qms.risk_free_rate(MARKET, SIGNAL, CFG)
MU, COV = qms.annualised_moments(RETS, CFG.cov_shrinkage)
W_MSR, NOTE = qms.max_sharpe_weights(RETS, RF, CFG)
W_MINVAR = qms.min_variance_weights(RETS, CFG)

r_msr, v_msr = qms.portfolio_return(W_MSR, MU), qms.portfolio_vol(W_MSR, COV)
r_mv, v_mv = qms.portfolio_return(W_MINVAR, MU), qms.portfolio_vol(W_MINVAR, COV)

print(f"as of {SIGNAL.date()} | {RETS.shape[1]} names, {len(RETS)} daily returns | solver: {NOTE}")
print(f"risk-free proxy ({CFG.rf_proxy}): {RF:.2%}")
print(f"  max-Sharpe   return {r_msr:>7.2%}   vol {v_msr:>6.2%}   Sharpe {(r_msr - RF) / v_msr:>5.2f}")
print(f"  min-variance return {r_mv:>7.2%}   vol {v_mv:>6.2%}   Sharpe {(r_mv - RF) / v_mv:>5.2f}")

### 3.1 · The efficient frontier and the capital market line

The frontier is traced by solving, for each target return, the lowest-variance portfolio
that reaches it under the same constraints the strategy uses — long-only, weights summing
to one, none above `max_weight`. The CML runs from the risk-free rate through the
tangency portfolio; the point where it touches the frontier is the max-Sharpe portfolio,
and the slope of that line *is* the Sharpe ratio.

Grey dots are the individual names. That every one of them sits below and to the right of
the frontier is the whole argument for optimising rather than equal-weighting.

The frontier is capped at `max_weight`, so it is shorter and flatter than a textbook one —
the constraint cuts off exactly the concentrated corner solutions an uncapped optimiser
would run to.

In [ ]:
targets = np.linspace(float(MU.min()), float(MU.max()), 60)
front = []
for t in targets:
    w = qms.min_variance_for_return(RETS, float(t), CFG)
    if w is not None:
        front.append((qms.portfolio_vol(w, COV), qms.portfolio_return(w, MU)))
front = pd.DataFrame(front, columns=["vol", "ret"]).sort_values("vol")
# Keep only the upper branch: below the min-variance point the same volatility is
# available at a higher return, so those portfolios are not efficient.
front = front[front["ret"] >= front.loc[front["vol"].idxmin(), "ret"]]

fig = figure(f"Efficient frontier — {SIGNAL.date()}{banner(MARKET)}",
             "Annualised volatility", "Annualised return", height=560)

asset_vol = np.sqrt(np.diag(COV))
fig.add_scatter(x=asset_vol, y=MU, mode="markers", name="Individual names",
                marker=dict(color="#c8c7c1", size=9, line=dict(color=SURFACE, width=1.5)),
                text=list(RETS.columns),
                hovertemplate="<b>%{text}</b><br>return %{y:.1%}<br>vol %{x:.1%}<extra></extra>")

fig.add_scatter(x=front["vol"], y=front["ret"], mode="lines", name="Efficient frontier",
                line=dict(color=SERIES["Strategy"], width=2.5),
                hovertemplate="frontier<br>return %{y:.1%}<br>vol %{x:.1%}<extra></extra>")

fig.add_scatter(x=[0, v_msr * 1.35], y=[RF, RF + (r_msr - RF) * 1.35], mode="lines",
                name="Capital market line", line=dict(color=MUTED, width=1.5, dash="dash"),
                hoverinfo="skip")

fig.add_scatter(x=[v_msr], y=[r_msr], mode="markers+text", name="Max-Sharpe (tangency)",
                marker=dict(color=SERIES["QQQ"], size=16, symbol="circle",
                            line=dict(color=SURFACE, width=2)),
                text=[f"  max-Sharpe {(r_msr - RF) / v_msr:.2f}"], textposition="middle right",
                textfont=dict(color=INK, size=11),
                hovertemplate="max-Sharpe<br>return %{y:.1%}<br>vol %{x:.1%}<extra></extra>")

fig.add_scatter(x=[v_mv], y=[r_mv], mode="markers+text", name="Minimum variance",
                marker=dict(color=SERIES["BOXX"], size=14, symbol="diamond",
                            line=dict(color=SURFACE, width=2)),
                text=["  min variance"], textposition="middle right",
                textfont=dict(color=INK, size=11),
                hovertemplate="min variance<br>return %{y:.1%}<br>vol %{x:.1%}<extra></extra>")

fig.add_scatter(x=[0], y=[RF], mode="markers+text", name=f"Risk-free ({CFG.rf_proxy})",
                marker=dict(color=MUTED, size=9), text=[f"  rf {RF:.1%}"],
                textposition="middle right", textfont=dict(color=MUTED, size=10),
                hoverinfo="skip")

fig.update_xaxes(tickformat=".0%", rangemode="tozero")
fig.update_yaxes(tickformat=".0%")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0))
fig.show()

### 3.2 · The weights, month by month

Drag the slider to walk the rebalances. Bars rather than a pie: sixteen slices cannot be
compared by eye, and the cap at `max_weight` is a vertical line you can actually read a
bar against.

In [ ]:
MIN_SHOWN = 0.005          # anything under 0.5% is rounding, not a position

frames, steps = [], []
for i, (date, wdict) in enumerate(zip(REBALANCES, PLAN["weights"])):
    s = pd.Series(wdict)
    s = s[s >= MIN_SHOWN].sort_values()
    frames.append(go.Frame(
        name=str(date.date()),
        data=[go.Bar(x=s.values, y=s.index, orientation="h",
                     marker=dict(color=SERIES["Strategy"], line=dict(color=SURFACE, width=2)),
                     text=[f"{v:.1%}" for v in s.values], textposition="outside",
                     textfont=dict(color=INK, size=11),
                     hovertemplate="<b>%{y}</b> %{x:.2%}<extra></extra>")],
        layout=go.Layout(title=dict(
            text=f"Weights on {date.date()} — {len(s)} positions, "
                 f"{PLAN['note'].iloc[i]}{banner(MARKET)}")),
    ))
    steps.append(dict(method="animate", label=date.strftime("%Y-%m"),
                      args=[[str(date.date())],
                            dict(mode="immediate", frame=dict(duration=0, redraw=True))]))

fig = figure("Weights", "Weight of the portfolio", "", height=560)
fig.add_traces(frames[0].data)
fig.frames = frames
fig.update_layout(
    title=frames[0].layout.title,
    xaxis=dict(tickformat=".0%", range=[0, max(CFG.max_weight, 0.3) * 1.22], gridcolor=GRID),
    yaxis=dict(gridcolor=SURFACE),
    sliders=[dict(active=0, steps=steps, x=0.06, len=0.94, pad=dict(t=40),
                  currentvalue=dict(prefix="Rebalance: ", font=dict(size=14, color=INK)))],
    updatemenus=[dict(type="buttons", direction="left", x=0.06, y=-0.18, xanchor="right",
                      showactive=False, buttons=[
                          dict(label="Play", method="animate",
                               args=[None, dict(frame=dict(duration=700, redraw=True), fromcurrent=True)]),
                          dict(label="Pause", method="animate",
                               args=[[None], dict(mode="immediate", frame=dict(duration=0, redraw=False))]),
                      ])],
)
fig.add_vline(x=CFG.max_weight, line=dict(color=SERIES["QQQ"], width=1.5, dash="dot"),
              annotation_text=f"cap {CFG.max_weight:.0%}", annotation_position="top",
              annotation_font=dict(color=SERIES["QQQ"], size=10))
fig.show()

### 3.3 · Every weight, every month, at once

The slider shows one month properly; this shows all of them at a glance. Magnitude, so one
hue: the darker the cell, the larger the position. Blank means not held.

Read it for two things — a name that stays dark across many months is a position the
optimiser keeps re-choosing, and a column of pale cells is a month where it spread the
book thin because nothing looked clearly better than anything else.

In [ ]:
weights = pd.DataFrame(list(PLAN["weights"]), index=[d.date() for d in REBALANCES]) * 100
weights = weights.loc[:, weights.max() >= MIN_SHOWN * 100]
weights = weights[weights.mean().sort_values(ascending=False).index]
shown = weights.where(weights >= MIN_SHOWN * 100)

fig = figure(f"Weight of each name at each rebalance (%){banner(MARKET)}", "", "",
             height=max(430, 21 * shown.shape[1]))
fig.add_heatmap(
    z=shown.T.values, x=[str(d) for d in shown.index], y=list(shown.columns),
    colorscale=BLUE_RAMP, xgap=2, ygap=2,
    colorbar=dict(title="weight %", thickness=12, outlinewidth=0, len=0.6),
    hovertemplate="%{y} — %{x}<br>%{z:.1f}%<extra></extra>",
)
fig.update_yaxes(type="category", autorange="reversed", tickmode="linear", dtick=1,
                 gridcolor=SURFACE, tickfont=dict(size=10))
fig.update_xaxes(type="category", tickmode="linear", dtick=1, gridcolor=SURFACE, tickangle=-45)
fig.show()

### 3.4 · What the optimiser was diversifying against

The correlation matrix of the selected names' daily returns, on the `LOOK_AT` date.
Polarity, so a diverging scale: blue is positive correlation, orange negative, neutral
grey at zero.

This is the check on whether the max-Sharpe answer means anything. Read the average
printed under the chart: on real Nasdaq-100 data expect something in the 0.4-0.7 range,
because twenty large-cap names drawn from the same few sectors move together. A
covariance matrix that uniform gives the optimiser very little genuine diversification to
find, so most of what it does is tilt toward the higher trailing means — the fragile part
of mean-variance. (A synthetic run shows roughly zero instead, because those paths are
generated independently; that is a property of the stand-in data, not of the strategy.)

In [ ]:
corr = RETS.corr()
order = corr.mean().sort_values(ascending=False).index      # cluster-ish, cheaply
corr = corr.loc[order, order]

fig = figure(f"Correlation of daily returns — trailing {len(RETS)} days to {SIGNAL.date()}"
             f"{banner(MARKET)}", "", "", height=max(520, 26 * len(corr)))
fig.add_heatmap(z=corr.values, x=list(corr.columns), y=list(corr.index),
                colorscale=DIVERGING, zmid=0, zmin=-1, zmax=1, xgap=1, ygap=1,
                colorbar=dict(title="ρ", thickness=12, outlinewidth=0, len=0.6),
                hovertemplate="%{y} vs %{x}<br>ρ = %{z:.2f}<extra></extra>")
fig.update_yaxes(type="category", autorange="reversed", tickmode="linear", dtick=1,
                 gridcolor=SURFACE, tickfont=dict(size=10))
fig.update_xaxes(type="category", tickmode="linear", dtick=1, gridcolor=SURFACE, tickangle=-45)
fig.show()

off_diagonal = corr.values[~np.eye(len(corr), dtype=bool)]
print(f"average pairwise correlation: {off_diagonal.mean():.2f}   "
      f"range {off_diagonal.min():.2f} to {off_diagonal.max():.2f}")

---

# 4 · Profit and loss

What the selection and the sizing added up to. Every curve starts at the $10,000
principal on the day before trading opens, so profit is profit on the money put in —
day one's move and the cost of getting in are inside the number, not before it.

In [ ]:
SUMMARY = pd.DataFrame({n: qms.equity_metrics(CURVES[n]) for n in CURVES.columns}).T
PNL = qms.pnl_by_symbol(TRADES, MARKET, EQUITY, CFG)

view = SUMMARY.copy()
for c in ["Total return", "CAGR", "Ann. vol", "Max drawdown", "Best day", "Worst day"]:
    view[c] = view[c].map(lambda v: f"{v:.2%}")
for c in ["Ending equity", "Profit"]:
    view[c] = view[c].map(lambda v: f"${v:,.0f}")
for c in ["Sharpe (rf=0)", "Calmar"]:
    view[c] = view[c].map(lambda v: f"{v:.2f}")

print(f"${CFG.principal:,.0f} from {CURVES.index[0].date()} to {CURVES.index[-1].date()}"
      f"{banner(MARKET)}")
view

### 4.1 · Portfolio value

The headline comparison. One axis, one currency, three series. Hover reads all three at
the same date, which is the question this chart exists to answer — not "what did the
strategy do" but "what did it do *instead of* the alternatives".

In [ ]:
fig = figure(f"$" + f"{CFG.principal:,.0f} invested {CURVES.index[0].date()}{banner(MARKET)}",
             "", "Portfolio value", height=520, hovermode="x unified")
for name in CURVES.columns:
    fig.add_scatter(x=CURVES.index, y=CURVES[name], mode="lines", name=name,
                    line=dict(color=SERIES[name], width=2.5),
                    hovertemplate=f"{name} $" + "%{y:,.0f}<extra></extra>")
    fig.add_annotation(x=CURVES.index[-1], y=CURVES[name].iloc[-1],
                       text=f"  {name} ${CURVES[name].iloc[-1]:,.0f}",
                       showarrow=False, xanchor="left", font=dict(color=INK, size=11))

fig.add_hline(y=CFG.principal, line=dict(color=MUTED, width=1, dash="dot"))
fig.update_yaxes(tickprefix="$", tickformat=",.0f")
fig.update_xaxes(range=[CURVES.index[0], CURVES.index[-1] + pd.Timedelta(days=len(CURVES) // 4)])
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0))
fig.show()

### 4.2 · Drawdown

The part a total-return figure hides. BOXX is on here to show what almost no drawdown
looks like beside the other two — the price of its flat equity curve is the absence of
this one.

In [ ]:
fig = figure(f"Drawdown from running peak{banner(MARKET)}", "", "Drawdown",
             height=400, hovermode="x unified")
for name in CURVES.columns:
    dd = CURVES[name] / CURVES[name].cummax() - 1.0
    fig.add_scatter(x=dd.index, y=dd, mode="lines", name=f"{name} (worst {dd.min():.1%})",
                    line=dict(color=SERIES[name], width=2),
                    hovertemplate=f"{name} " + "%{y:.2%}<extra></extra>")
    fig.add_scatter(x=[dd.idxmin()], y=[dd.min()], mode="markers", showlegend=False,
                    marker=dict(color=SERIES[name], size=10, line=dict(color=SURFACE, width=2)),
                    hoverinfo="skip")
fig.add_hline(y=0, line=dict(color=GRID, width=1))
fig.update_yaxes(tickformat=".0%")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0))
fig.show()

### 4.3 · How the $10,000 got where it did

The strategy's month-by-month profit in dollars, as a waterfall from the principal to the
ending balance. Blue months added money, orange months lost it; the final bar is the
total. Dollars rather than percentages, because a 3% month early on and a 3% month late
are not the same amount of money and the waterfall is the chart that admits it.

In [ ]:
month_end = pd.concat([CURVES.iloc[[0]], CURVES.resample("ME").last()])["Strategy"]
month_pnl = month_end.diff().dropna()

fig = figure(f"Monthly profit and loss, strategy{banner(MARKET)}", "", "Profit", height=470)
fig.add_trace(go.Waterfall(
    orientation="v",
    measure=["relative"] * len(month_pnl) + ["total"],
    x=[d.strftime("%Y-%m") for d in month_pnl.index] + ["Total"],
    y=list(month_pnl.values) + [0],
    text=[f"${v:,.0f}" for v in month_pnl.values] + [f"${month_end.iloc[-1] - CFG.principal:,.0f}"],
    textposition="outside", textfont=dict(size=10, color=INK),
    increasing=dict(marker=dict(color=POS, line=dict(color=SURFACE, width=2))),
    decreasing=dict(marker=dict(color=NEG, line=dict(color=SURFACE, width=2))),
    totals=dict(marker=dict(color=MUTED, line=dict(color=SURFACE, width=2))),
    connector=dict(line=dict(color=GRID, width=1)),
    hovertemplate="%{x}<br>$%{y:,.0f}<extra></extra>",
))
fig.update_yaxes(tickprefix="$", tickformat=",.0f")
# Category, not date: the last bar is labelled "Total", and a date axis silently drops it.
fig.update_xaxes(type="category", tickangle=-45, gridcolor=SURFACE)
fig.update_layout(showlegend=False)
fig.show()

wins = int((month_pnl > 0).sum())
print(f"{wins} of {len(month_pnl)} months profitable | "
      f"best ${month_pnl.max():,.0f} ({month_pnl.idxmax():%Y-%m}) | "
      f"worst ${month_pnl.min():,.0f} ({month_pnl.idxmin():%Y-%m})")

### 4.4 · Which names made the money

Profit per symbol, computed from the fills alone: cash out minus cash in, plus whatever
the position was still worth at the end. The column sums to the strategy's total profit —
the reconciliation is printed below the chart — so this attribution cannot quietly
disagree with the equity curve above it.

Polarity again, so blue against orange with the zero line as the neutral midpoint. This is
usually the most uncomfortable chart in the set: a monthly-rebalanced twenty-name book
tends to owe most of its result to two or three names, which is a concentration the
diversified-looking weights do not advertise.

In [ ]:
pnl = PNL.sort_values("pnl")
total = float(pnl["pnl"].sum())

fig = figure(f"Profit and loss by name{banner(MARKET)}", "Profit", "",
             height=max(430, 17 * len(pnl)))
fig.add_bar(
    x=pnl["pnl"], y=pnl["symbol"], orientation="h",
    marker=dict(color=[POS if v >= 0 else NEG for v in pnl["pnl"]],
                line=dict(color=SURFACE, width=1.5)),
    customdata=pnl[["traded_notional", "costs", "end_value"]],
    hovertemplate=("<b>%{y}</b><br>P&L $%{x:,.0f}"
                   "<br>traded $%{customdata[0]:,.0f}"
                   "<br>commission $%{customdata[1]:,.2f}"
                   "<br>still held $%{customdata[2]:,.0f}<extra></extra>"),
)
fig.add_vline(x=0, line=dict(color=MUTED, width=1))
fig.update_xaxes(tickprefix="$", tickformat=",.0f")
fig.update_yaxes(type="category", tickmode="linear", dtick=1, gridcolor=SURFACE,
                 tickfont=dict(size=10))
fig.update_layout(showlegend=False)
fig.show()

strategy_profit = float(CURVES["Strategy"].iloc[-1] - CFG.principal)
top3 = pnl.nlargest(3, "pnl")
print(f"sum of per-name P&L  ${total:,.2f}")
print(f"strategy profit      ${strategy_profit:,.2f}   "
      f"(difference ${abs(total - strategy_profit):,.2f})")
print(f"\ntop 3 names are ${top3['pnl'].sum():,.0f} of ${total:,.0f} "
      f"({top3['pnl'].sum() / total:.0%} of the profit): {', '.join(top3['symbol'])}")
print(f"{int((pnl['pnl'] > 0).sum())} of {len(pnl)} names profitable | "
      f"total commission ${pnl['costs'].sum():,.2f}")

### 4.5 · Strategy against the baselines, month by month

The same months as the waterfall, in percent, beside QQQ and BOXX. Percent here rather
than dollars because the three lines started with the same $10,000 but did not stay the
same size, and the question this chart answers is which one had the better month.

In [ ]:
monthly = pd.concat([CURVES.iloc[[0]], CURVES.resample("ME").last()]).pct_change().dropna(how="all")

fig = figure(f"Monthly return{banner(MARKET)}", "", "Return", height=430, barmode="group")
for name in CURVES.columns:
    fig.add_bar(x=[d.strftime("%Y-%m") for d in monthly.index], y=monthly[name], name=name,
                marker=dict(color=SERIES[name], line=dict(color=SURFACE, width=2)),
                hovertemplate=f"{name} " + "%{y:.2%}<extra></extra>")
fig.add_hline(y=0, line=dict(color=MUTED, width=1))
fig.update_yaxes(tickformat=".0%")
fig.update_xaxes(tickangle=-45, gridcolor=SURFACE)
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0))
fig.show()

beat = {n: int((monthly["Strategy"] > monthly[n]).sum()) for n in CURVES.columns if n != "Strategy"}
print(" | ".join(f"strategy beat {n} in {v} of {len(monthly)} months" for n, v in beat.items()))
(monthly * 100).round(2)

---

## 5 · What to change

Everything above is driven by two variables. Re-run from the cell that sets each.

* **`LOOK_AT`** (section 2) — which rebalance the selection, frontier, weights and
  correlation charts describe. `0` is the first month, `-1` the most recent.
* **`CFG`** (section 1) — the strategy itself. Changing it means re-running
  `run_strategy`, which is the slow cell.

```python
CFG = qms.BacktestSettings(top_n=10)          # fewer, larger positions
CFG = qms.BacktestSettings(max_weight=1.0)    # uncapped — watch the frontier lengthen
CFG = qms.BacktestSettings(cov_shrinkage=0.0) # raw sample covariance
CFG = qms.BacktestSettings(rf_proxy=None)     # rf = 0 in the Sharpe objective
```

Two cautions carried over from the back-test notebook, because they apply just as much to
a chart as to a table:

* The universe is **today's** Nasdaq-100 held fixed across the window, so the selection
  charts show names that were not always in QQQ. This biases the strategy's result upward.
* Fifteen months is one path through one regime. These pictures describe what happened.
  They are not evidence about what happens next, and the frontier in particular is drawn
  from trailing estimates that the next month is under no obligation to honour.